# Step 2: MobileNetV4ConvSmall Training, Export, and Inference

This notebook trains the SoundPulse Tiny CNN replacement using TensorFlow's official `MobileNetV4ConvSmall` backbone. The notebook keeps data loading, feature generation, model definition, training, prototype extraction, TFLite export, and sample inference in one place.

In [ ]:
%pip install -q tensorflow tf-models-official pandas numpy librosa soundfile scikit-learn matplotlib seaborn

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import librosa
import numpy as np
import pandas as pd
import soundfile as sf
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix
from official.vision.modeling.backbones import mobilenet

PROJECT_ROOT = Path.cwd().resolve().parents[1]
ML_ROOT = PROJECT_ROOT / 'ml'
CONTRACTS_DIR = PROJECT_ROOT / 'packages' / 'shared_contracts' / 'v1'
CURATED_MANIFEST_PATH = ML_ROOT / 'data' / 'curated' / 'soundpulse_manifest.v1.json'
MODEL_OUTPUT_DIR = ML_ROOT / 'export' / 'artifacts' / 'generated' / 'mobilenetv4_convsmall'
MODEL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

with (CONTRACTS_DIR / 'class_labels.json').open('r', encoding='utf-8') as handle:
    CLASS_CONTRACT = json.load(handle)
with (CONTRACTS_DIR / 'preprocessing_config.json').open('r', encoding='utf-8') as handle:
    PREPROCESSING_CONTRACT = json.load(handle)

CLASS_ORDER = [item['slug'] for item in CLASS_CONTRACT['classes']]
CLASS_TO_INDEX = {slug: index for index, slug in enumerate(CLASS_ORDER)}
SAMPLE_RATE = PREPROCESSING_CONTRACT['audio']['sample_rate_hz']
WINDOW_DURATION_MS = PREPROCESSING_CONTRACT['audio']['window_duration_ms']
WINDOW_SAMPLES = int(SAMPLE_RATE * WINDOW_DURATION_MS / 1000)
IMAGE_SIZE = (224, 224)
EMBEDDING_DIM = PREPROCESSING_CONTRACT['embedding_contract']['tiny_cnn_dim']
BATCH_SIZE = 16
EPOCHS = 20
AUTOTUNE = tf.data.AUTOTUNE

assert CURATED_MANIFEST_PATH.exists(), 'Run Notebook 1 first to build the curated manifest.'

print(f'Manifest path: {CURATED_MANIFEST_PATH}')
print(f'Class order: {CLASS_ORDER}')

In [ ]:
with CURATED_MANIFEST_PATH.open('r', encoding='utf-8') as handle:
    manifest = json.load(handle)

records = []
for dataset in manifest['datasets']:
    dataset_root = PROJECT_ROOT / 'external_data' / dataset['name']
    for clip in dataset['clips']:
        records.append({
            'dataset_name': dataset['name'],
            'audio_path': str(dataset_root.parent / clip['relative_path']),
            'class_slug': clip['class_slug'],
            'split': clip['split'],
            'clip_id': clip['clip_id'],
            'source_group_id': clip['source_group_id'],
        })

manifest_df = pd.DataFrame(records)
manifest_df.head()

In [ ]:
def load_audio(audio_path: tf.Tensor) -> tf.Tensor:
    path = audio_path.numpy().decode('utf-8')
    waveform, sample_rate = sf.read(path, dtype='float32', always_2d=False)
    if waveform.ndim > 1:
        waveform = np.mean(waveform, axis=1)
    if sample_rate != SAMPLE_RATE:
        waveform = librosa.resample(waveform, orig_sr=sample_rate, target_sr=SAMPLE_RATE)
    if waveform.shape[0] < WINDOW_SAMPLES:
        waveform = np.pad(waveform, (0, WINDOW_SAMPLES - waveform.shape[0]))
    else:
        waveform = waveform[:WINDOW_SAMPLES]
    return waveform.astype(np.float32)


def waveform_to_rgb_log_mel(waveform: tf.Tensor) -> tf.Tensor:
    stft = tf.signal.stft(
        waveform,
        frame_length=PREPROCESSING_CONTRACT['stft']['win_length'],
        frame_step=PREPROCESSING_CONTRACT['stft']['hop_length'],
        fft_length=PREPROCESSING_CONTRACT['stft']['n_fft'],
        window_fn=tf.signal.hann_window,
        pad_end=False,
    )
    spectrogram = tf.abs(stft) ** PREPROCESSING_CONTRACT['mel']['power']
    mel_matrix = tf.signal.linear_to_mel_weight_matrix(
        num_mel_bins=PREPROCESSING_CONTRACT['mel']['n_mels'],
        num_spectrogram_bins=spectrogram.shape[-1],
        sample_rate=SAMPLE_RATE,
        lower_edge_hertz=PREPROCESSING_CONTRACT['mel']['f_min_hz'],
        upper_edge_hertz=PREPROCESSING_CONTRACT['mel']['f_max_hz'],
    )
    mel_spectrogram = tf.matmul(spectrogram, mel_matrix)
    log_mel = tf.math.log(mel_spectrogram + PREPROCESSING_CONTRACT['mel']['log_offset'])
    log_mel = tf.clip_by_value(
        log_mel,
        PREPROCESSING_CONTRACT['normalization']['clip_min_db'],
        PREPROCESSING_CONTRACT['normalization']['clip_max_db'],
    )
    log_mel = tf.expand_dims(log_mel, axis=-1)
    log_mel = tf.image.resize(log_mel, IMAGE_SIZE)
    log_mel = tf.repeat(log_mel, repeats=3, axis=-1)
    log_mel = tf.image.per_image_standardization(log_mel)
    return log_mel


def load_example(audio_path: tf.Tensor, label: tf.Tensor):
    waveform = tf.py_function(load_audio, [audio_path], Tout=tf.float32)
    waveform.set_shape([WINDOW_SAMPLES])
    image = waveform_to_rgb_log_mel(waveform)
    image.set_shape([IMAGE_SIZE[0], IMAGE_SIZE[1], 3])
    return image, label

In [ ]:
def make_dataset(split_name: str, training: bool) -> tf.data.Dataset:
    frame = manifest_df[manifest_df['split'] == split_name].copy()
    labels = frame['class_slug'].map(CLASS_TO_INDEX).astype(np.int32).to_numpy()
    dataset = tf.data.Dataset.from_tensor_slices((frame['audio_path'].to_numpy(), labels))
    if training:
        dataset = dataset.shuffle(buffer_size=len(frame), seed=42, reshuffle_each_iteration=True)
    dataset = dataset.map(load_example, num_parallel_calls=AUTOTUNE)
    dataset = dataset.batch(BATCH_SIZE).prefetch(AUTOTUNE)
    return dataset


train_ds = make_dataset('train', training=True)
validation_ds = make_dataset('validation', training=False)
test_ds = make_dataset('test', training=False)

train_ds

In [ ]:
def build_mobilenetv4_convsmall_classifier(num_classes: int) -> tuple[tf.keras.Model, tf.keras.Model]:
    inputs = tf.keras.Input(shape=(IMAGE_SIZE[0], IMAGE_SIZE[1], 3), name='log_mel_rgb')
    backbone = mobilenet.MobileNet(
        model_id='MobileNetV4ConvSmall',
        filter_size_scale=1.0,
        input_specs=tf.keras.layers.InputSpec(shape=[None, IMAGE_SIZE[0], IMAGE_SIZE[1], 3]),
        output_intermediate_endpoints=True,
    )
    endpoints = backbone(inputs)
    last_level = max(endpoints.keys(), key=int)
    features = endpoints[last_level]
    pooled = tf.keras.layers.GlobalAveragePooling2D(name='global_avg_pool')(features)
    embedding = tf.keras.layers.Dense(EMBEDDING_DIM, name='embedding')(pooled)
    normalized_embedding = tf.keras.layers.Lambda(
        lambda tensor: tf.math.l2_normalize(tensor, axis=-1),
        name='embedding_norm',
    )(embedding)
    outputs = tf.keras.layers.Dense(num_classes, activation='softmax', name='classifier')(normalized_embedding)
    classifier = tf.keras.Model(inputs=inputs, outputs=outputs, name='soundpulse_mobilenetv4_convsmall_classifier')
    embedding_model = tf.keras.Model(inputs=inputs, outputs=normalized_embedding, name='soundpulse_mobilenetv4_convsmall_embedding_model')
    return classifier, embedding_model


classifier, embedding_model = build_mobilenetv4_convsmall_classifier(len(CLASS_ORDER))
classifier.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)
classifier.summary()

In [ ]:
callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(MODEL_OUTPUT_DIR / 'best.weights.h5'),
        monitor='val_accuracy',
        save_best_only=True,
        save_weights_only=True,
    ),
    tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True),
]

history = classifier.fit(
    train_ds,
    validation_data=validation_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
)

In [ ]:
test_probabilities = classifier.predict(test_ds)
test_predictions = np.argmax(test_probabilities, axis=1)
test_labels = np.concatenate([labels.numpy() for _, labels in test_ds], axis=0)

print(classification_report(test_labels, test_predictions, target_names=CLASS_ORDER, zero_division=0))
confusion_matrix(test_labels, test_predictions)

In [ ]:
def collect_embeddings(dataset: tf.data.Dataset) -> tuple[np.ndarray, np.ndarray]:
    images = []
    labels = []
    for batch_images, batch_labels in dataset:
        images.append(embedding_model.predict(batch_images, verbose=0))
        labels.append(batch_labels.numpy())
    return np.concatenate(images, axis=0), np.concatenate(labels, axis=0)


train_embeddings, train_labels = collect_embeddings(train_ds)
prototype_store = {}
for class_index, class_slug in enumerate(CLASS_ORDER):
    mask = train_labels == class_index
    if mask.any():
        prototype_store[class_slug] = train_embeddings[mask].mean(axis=0).tolist()

with (MODEL_OUTPUT_DIR / 'prototype_store.json').open('w', encoding='utf-8') as handle:
    json.dump(prototype_store, handle, indent=2)

print(f'Prototype classes exported: {sorted(prototype_store.keys())}')

In [ ]:
embedding_model.export(str(MODEL_OUTPUT_DIR / 'saved_model'))
converter = tf.lite.TFLiteConverter.from_saved_model(str(MODEL_OUTPUT_DIR / 'saved_model'))
tflite_model = converter.convert()
with (MODEL_OUTPUT_DIR / 'mobilenetv4_convsmall_embedding.tflite').open('wb') as handle:
    handle.write(tflite_model)

metadata = {
    'model_name': 'MobileNetV4ConvSmall',
    'embedding_dim': EMBEDDING_DIM,
    'class_order': CLASS_ORDER,
    'image_size': IMAGE_SIZE,
    'sample_rate_hz': SAMPLE_RATE,
}
with (MODEL_OUTPUT_DIR / 'export_metadata.json').open('w', encoding='utf-8') as handle:
    json.dump(metadata, handle, indent=2)

print(f'Exported TFLite embedding model to {MODEL_OUTPUT_DIR}')

In [ ]:
sample_row = manifest_df.iloc[0]
sample_waveform = load_audio(tf.constant(sample_row['audio_path']))
sample_image = waveform_to_rgb_log_mel(tf.convert_to_tensor(sample_waveform))
sample_embedding = embedding_model.predict(tf.expand_dims(sample_image, axis=0), verbose=0)[0]

distances = {
    class_slug: float(np.sum((sample_embedding - np.array(prototype)) ** 2))
    for class_slug, prototype in prototype_store.items()
}
predicted_label = min(distances, key=distances.get)
print({'clip_id': sample_row['clip_id'], 'true_label': sample_row['class_slug'], 'predicted_label': predicted_label, 'distance': distances[predicted_label]})